In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# Load raw data
df = pd.read_csv('for_plos.csv')

# Map choices to binary integers
df['choice_binary'] = df['key'].map({"R1": 0, "R2": 1}).astype(int)

# Integer-code participant IDs for consistent ordering
df["subid"] = df["ID"].astype("category").cat.codes

unique_ids    = sorted(df['subid'].unique())
unique_blocks = sorted(df['block'].unique())
n_subjects    = len(unique_ids)
n_blocks      = len(unique_blocks)

# Compute maximum block length
block_lengths_all = []
for sub in unique_ids:
    sub_df = df[df['subid'] == sub]
    for block in unique_blocks:
        block_df = sub_df[sub_df['block'] == block]
        block_lengths_all.append(len(block_df))

max_block_len = int(np.max(block_lengths_all))
input_dim = 2   # [prev_choice, prev_reward]

print(f"n_subjects={n_subjects}, n_blocks={n_blocks}, "
      f"max_block_len={max_block_len}")

# Build arrays
# Shape: (n_subjects, n_blocks, max_block_len, input_dim)
# Padding value -100: cross-entropy ignores targets of -100 by default.
xin           = np.full((n_subjects, n_blocks, max_block_len, input_dim), -100.0, dtype=np.float32)
c             = np.full((n_subjects, n_blocks, max_block_len),            -100.0, dtype=np.float32)
choice_one_hot = np.zeros((n_subjects, n_blocks, max_block_len, 2),               dtype=np.float32)
diagnosis     = np.empty(n_subjects, dtype=object)

for sub_idx, sub in enumerate(unique_ids):
    sub_df = df[df['subid'] == sub]
    diagnosis[sub_idx] = sub_df['diag'].iloc[0]

    for b_idx, block in enumerate(unique_blocks):
        block_df = sub_df[sub_df['block'] == block].reset_index(drop=True)
        if len(block_df) == 0:
            continue

        L       = len(block_df)
        choices = block_df['choice_binary'].values   # int 0/1
        rewards = block_df['reward'].values          # 0/1

        # First trial of the block: no previous history → input zeros
        xin[sub_idx, b_idx, 0, 0] = 0.0
        xin[sub_idx, b_idx, 0, 1] = 0.0

        # Remaining trials: previous trial's choice & reward as input
        if L > 1:
            xin[sub_idx, b_idx, 1:L, 0] = choices[:L - 1].astype(np.float32)
            xin[sub_idx, b_idx, 1:L, 1] = rewards[:L - 1].astype(np.float32)

        # Targets: choice at each trial (padding positions stay -100)
        c[sub_idx, b_idx, :L]    = choices.astype(np.float32)
        choice_one_hot[sub_idx, b_idx, :L, 0] = (choices == 0).astype(np.float32)
        choice_one_hot[sub_idx, b_idx, :L, 1] = (choices == 1).astype(np.float32)

print(f"xin shape: {xin.shape}")
print(f"c shape: {c.shape}, values: {np.unique(c)}")

# Build per-subject metadata DataFrame
df_all = pd.DataFrame({
    'subid': unique_ids,
    'diag':  [diagnosis[i] for i in range(n_subjects)],
})
print(f"Diagnosis groups: {sorted(df_all['diag'].unique())}")

sub_to_row = {s: i for i, s in enumerate(unique_ids)}

def _idx(subjects):
    return [sub_to_row[s] for s in sorted(subjects) if s in sub_to_row]


# Train / val / test split (no val → empty arrays)
def split_subjects(subject_ids, train_ratio=0.7, test_ratio=0.3, seed=42):
    rng = np.random.default_rng(seed)
    ids = np.array(sorted(subject_ids))
    rng.shuffle(ids)
    n_train = int(len(ids) * train_ratio)
    return set(ids[:n_train]), set(), set(ids[n_train:])


train_subjects, val_subjects, test_subjects = split_subjects(unique_ids)
tr_idx = _idx(train_subjects)
te_idx = _idx(test_subjects)

xin_train = xin[tr_idx]
c_train = c[tr_idx]
choice_one_hot_train = choice_one_hot[tr_idx]

xin_test = xin[te_idx]
c_test = c[te_idx]
choice_one_hot_test = choice_one_hot[te_idx]

print(f"Train: {len(tr_idx)} subjects, Test: {len(te_idx)} subjects")
print(xin_train.shape)
print(c_train.shape)

n_subjects=101, n_blocks=12, max_block_len=202
xin shape: (101, 12, 202, 2)
c shape: (101, 12, 202), values: [-100.    0.    1.]
Diagnosis groups: ['Bipolar', 'Depression', 'Healthy']
Train: 70 subjects, Test: 31 subjects
(70, 12, 202, 2)
(70, 12, 202)


In [ ]:
# build the RNN
import torch
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"

#define hyperparameters
hidden_size = 10
num_layers = 1
input_size = 2
num_actions = 2
learning_rate = 1e-03

class RNN_LSTM(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers, num_actions):
    super(RNN_LSTM, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True) #restructure data accordingly
    self.fc = nn.Linear(hidden_size, num_actions)

  def forward(self, x):
    h0=torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
    c0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size,device=x.device)
    out, _ = self.lstm(x, (h0, c0))
    logits = self.fc(out)
    return logits

class RNN_GRU(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers, num_actions):
    super(RNN_GRU, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True) #restructure data accordingly
    self.fc = nn.Linear(hidden_size, num_actions)

  def forward(self, x):
    h0=torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
    out, _ = self.gru(x, h0)
    logits = self.fc(out)
    return logits

X = xin_train.reshape(-1, xin_train.shape[2], xin_train.shape[3])
y = c_train.reshape(-1, c_train.shape[2])

X_test = xin_test.reshape(-1, xin_test.shape[2], xin_test.shape[3])
y_test = c_test.reshape(-1, c_test.shape[2])

dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
train_data = DataLoader(dataset, batch_size=32, shuffle=True)

dataset_test = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
test_data = DataLoader(dataset_test, batch_size=32, shuffle=False)

#training
num_epochs = 500
model = RNN_GRU(input_size, hidden_size, num_layers, 2).to(device)

# define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

def train_epoch(model, loader, optimizer, criterion, device, accuracy=True):
  model.train()

  train_loss = 0
  num_correct = 0
  num_samples = 0

  for x_batch, y_batch in loader:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    logits = model(x_batch)
    loss = criterion(logits.reshape(-1, logits.shape[-1]), y_batch.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
    if accuracy:
      preds = logits.argmax(dim=-1)
      mask = y_batch != -100 # because of the padding
      num_correct += (preds[mask] == y_batch[mask]).sum().item()
      num_samples += mask.sum().item()
  return {"loss": train_loss / len(loader),
          "accuracy": num_correct/num_samples}

def evaluate(model, loader, criterion, device):
  model.eval()

  test_loss = 0
  num_correct = 0
  num_samples = 0

  with torch.no_grad():
    for x_test_batch, y_test_batch in loader:
      x_test_batch, y_test_batch = x_test_batch.to(device), y_test_batch.to(device)
      logits_test = model(x_test_batch)
      loss_test = criterion(logits_test.reshape(-1, logits_test.shape[-1]), y_test_batch.reshape(-1))
      test_loss += loss_test.item()
      preds = logits_test.argmax(dim=-1)

      mask = y_test_batch != -100

      num_correct += (preds[mask] == y_test_batch[mask]).sum().item()
      num_samples += mask.sum().item()
  return {
        "loss": test_loss / len(loader),
        "accuracy": num_correct / num_samples}


# actual training run
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []
for epoch in tqdm(range(num_epochs)):

  train_metrics = train_epoch(model, train_data, optimizer, criterion, device)
  test_metrics = evaluate(model, test_data, criterion, device)
  train_losses.append(train_metrics["loss"])
  test_losses.append(test_metrics["loss"])
  train_accuracies.append(train_metrics["accuracy"])
  test_accuracies.append(test_metrics["accuracy"])

  print(
    f"Epoch {epoch+1}/{num_epochs}, "
    f"Train Loss: {train_metrics['loss']:.4f}, "
    f"Test Loss: {test_metrics['loss']:.4f}, "
    f"Train Accuracy: {train_metrics['accuracy']:.4f}, "
    f"Test Accuracy: {test_metrics['accuracy']:.4f}"
)





  0%|          | 1/1500 [00:02<50:52,  2.04s/it]

Epoch 1/1500, Train Loss: 0.6540, Test Loss: 0.6405, Train Accuracy: 0.7771, Test Accuracy: 0.7476


  0%|          | 2/1500 [00:03<44:45,  1.79s/it]

Epoch 2/1500, Train Loss: 0.6245, Test Loss: 0.6060, Train Accuracy: 0.7609, Test Accuracy: 0.7477


  0%|          | 3/1500 [00:05<42:56,  1.72s/it]

...

Epoch 229/1500, Train Loss: 0.3020, Test Loss: 0.2699, Train Accuracy: 0.8750, Test Accuracy: 0.9003


 15%|█▌        | 230/1500 [06:53<36:46,  1.74s/it]

Epoch 230/1500, Train Loss: 0.3007, Test Loss: 0.2696, Train Accuracy: 0.8752, Test Accuracy: 0.9003


 15%|█▌        | 230/1500 [06:55<38:12,  1.81s/it]


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
plt.plot(train_losses, test_losses)

NameError: name 'train_losses' is not defined

In [ ]:
# not used
# Check accuracy on training & test to see how good our model
def check_accuracy(loader, model, train=True):
    num_correct = 0
    num_samples = 0

    # Set model to eval
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            logits = model(x)
            predictions = logits.argmax(dim=2)
            mask = y != -100
            num_correct += (predictions[mask] == y[mask]).sum().item()
            num_samples += mask.sum().item()

    # Toggle model back to train if called during training
    if train:
      model.train()
    return num_correct / num_samples